In [1]:
# Install the Ultralytics YOLOv8 library and Roboflow package
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="5l9MwdgK1oJS4w8xVyaF")
project = rf.workspace("honilik148-hideam-com").project("cortex-mega-project")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Cortex-Mega-Project-1 in yolov8:: 100%|██████████| 73958/73958 [00:15<00:00, 4682.93it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
from ultralytics import YOLO

# 1. Load the pre-trained YOLOv8 Nano model (optimized for Edge/Raspberry Pi)
model = YOLO('yolov8n.pt')

# 2. Train the model on your newly downloaded dataset
# dataset.location automatically finds the data.yaml file from Cell 2
results = model.train(
    data=f"{dataset.location}/data.yaml",
        epochs=30,
            imgsz=640,
                batch=64,# Optimal batch size for Colab's free T4 GPU
    workers=8,
    fraction=0.25,
                    project="Cortex",   # Folder name where results are saved
                        name="mega_model"   # Subfolder for this specific training run
                        )

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Cortex-Mega-Project-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.25, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mega_model, nbs=64, nms=None, opset

In [4]:
# Export the best weights to ONNX format for the Raspberry Pi
# half=False ensures standard 32-bit floating point math for ARM CPU compatibility
export_path = model.export(format="onnx", imgsz=640, half=False)

print(f"Model exported successfully to: {export_path}")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/Cortex/mega_model/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 868ms
Prepared 5 packages in 1.01s
Uninstalled 1 package in 9ms
Installed 5 packages in 39ms
 + colorama==0.4.6
 + onnx==1.23.0
 + onnxruntime==1.30.0
 + onnxslim==0.1.96
 - protobuf==5.29.6
 + protobuf==7.36.2

requirements: AutoUpdate s

In [6]:
from google.colab import files

# Download the specific ONNX file generated by the training run
files.download('/content/runs/detect/Cortex/mega_model/weights/best.onnx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import shutil
from google.colab import files

# 1. Compress the entire 'Cortex' project folder into a zip file
# (If your logs saved to the default runs folder instead, change '/content/Cortex' to '/content/runs')
shutil.make_archive('cortex_training_results', 'zip', '/content/runs/detect/Cortex')

# 2. Trigger the download to your laptop
files.download('cortex_training_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>